In [1]:
!pip install playsound
!pip install sounddevice scipy
!pip install SpeechRecognition
!pip install pipwin
!pipwin install pyaudio

/Users/hsoo3844/anaconda3/lib/python3.11/site-packages/pipwin/command.py:66: UserWarning: Found a non Windows system. Package installation might not work.
  warn("Found a non Windows system. Package installation might not work.")
Package `pyaudio` found in cache
https://download.lfd.uci.edu/pythonlibs/archived/PyAudio-0.2.11-cp311-cp311-win_amd64.whl
PyAudio-0.2.11-cp311-cp311-win_amd64.whl
File /Users/hsoo3844/pipwin/PyAudio-0.2.11-cp311-cp311-win_amd64.whl already exists
ERROR: PyAudio-0.2.11-cp311-cp311-win_amd64.whl is not a supported wheel on this platform.
Traceback (most recent call last):
  File "/Users/hsoo3844/anaconda3/bin/pipwin", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/Users/hsoo3844/anaconda3/lib/python3.11/site-packages/pipwin/command.py", line 103, in main
    cache.install(package)
  File "/Users/hsoo3844/anaconda3/lib/python3.11/site-packages/pipwin/pipwin.py", line 301, in install
    subprocess.check_call([executable, "-m", "pip", "inst

In [1]:
import os
import openai
import requests
import json
import playsound
import speech_recognition as sr
from gtts import gTTS  # gTTS install 필요
import time  # 시간 제한을 위한 모듈

# voice to text
def voice2text():
    try:
        r = sr.Recognizer()

        with sr.Microphone() as source:
            print("음성을 입력해주세요.")
            print("종료를 원하실 경우, *종료*라고 말해주세요.")
            audio = r.record(source, duration=4) # 입력 제한시간 4초(duration = 4)
            try:
                text = r.recognize_google(audio, language='ko-KR')
                print('입력 : ' + text)
                return text.split()[0], text

            except sr.UnknownValueError:
                print('오디오를 이해할 수 없습니다.')
                return None
            
            except sr.RequestError as e:
                print(f'에러가 발생하였습니다.\n에러원인 : {e}')
                return None
                
    except KeyboardInterrupt:
        return None

# text to voice
def text2voice(text, title):
    try:
        # TTS 객체 생성
        tts = gTTS(text=text, lang='ko')

        # mp3 파일로 저장
        tts.save("result/" + title + "_답변.mp3")
        print(f"출력: {text}")
        return True

    except Exception as e:
        print(f"text2voice 실행 중 오류 발생: {str(e)}")
        return False


def chatBot():
    try:
        with open('key.txt', 'r') as file:
            key = file.read()
    except:
        print(f"비밀키 로드 실패.\nERROR: {str(e)}")
        return False
    try:
        start_time = time.time()  # 시작 시간 저장

        # OpenAI API 키 설정
        openai.api_key = key

        # 대화 내용을 저장할 리스트
        conversation = [
            {"role": "system", "content": "You are a helpful assistant."},
        ]

        while True:  # 대화를 계속 이어갈 수 있도록 while loop 추가
            voice_result = voice2text()
            for _ in range(2):        # 입력에 오류가 있을 때 다시 입력 받음.
                if not voice_result:
                    voice_result = voice2text()
                else:
                    title, text = voice_result
                    break
                    
            # 사용자의 메시지 추가
            conversation.append({"role": "user", "content": text})

            # 챗봇 세션 시작
            response = openai.ChatCompletion.create(
                model="gpt-3.5-turbo",  # 사용할 모델 설정
                messages=conversation
            )

            r_text = response['choices'][0]['message']['content']
            # assistant의 메시지 추가
            conversation.append({"role": "assistant", "content": r_text})

            text2voice(r_text, title)
            playsound.playsound("./result/" + title + "_답변.mp3")

            # 특정 키워드를 입력받으면 종료
            if '종료' in text:
                print("챗봇이 종료되었습니다.")
                break

            # 200초가 지나면 종료
            if time.time() - start_time > 200:
                print("시간이 지나 챗봇이 종료되었습니다.")
                break

        return True

    except Exception as e:
        print(f"챗봇 실행 중 오류 발생하여 종료합니다.\nERROR: {str(e)}")
        return False


if __name__ == '__main__':
    if chatBot():
        print("챗봇 실행 완료.")
    else:
        print("챗봇 실행 실패.")


음성을 입력해주세요.
종료를 원하실 경우, *종료*라고 말해주세요.
오디오를 이해할 수 없습니다.
음성을 입력해주세요.
종료를 원하실 경우, *종료*라고 말해주세요.
오디오를 이해할 수 없습니다.
음성을 입력해주세요.
종료를 원하실 경우, *종료*라고 말해주세요.
오디오를 이해할 수 없습니다.
챗봇 실행 중 오류 발생하여 종료합니다.
ERROR: cannot access local variable 'text' where it is not associated with a value
챗봇 실행 실패.
